# **Proyek Machine Learning - Sistem Rekomendasi Wisata Jawa Barat**

Proyek ini membangun sistem rekomendasi tempat wisata di Jawa Barat menggunakan pendekatan **Content-Based Filtering** berbasis algoritma **TF-IDF** dan **Cosine Similarity**. Sistem akan memberikan rekomendasi wisata yang paling mirip berdasarkan kemiripan konten deskripsi dan kategori wisata.

## **1. Import Library**

Pada langkah pertama, semua library yang diperlukan diimpor. `pandas` digunakan untuk manipulasi data tabular, `matplotlib` dan `seaborn` untuk membuat visualisasi data, `re` untuk pemrosesan teks berbasis regular expression, serta `pickle` untuk menyimpan model yang sudah dilatih. Dari `sklearn`, digunakan `TfidfVectorizer` untuk ekstraksi fitur teks dan `cosine_similarity` untuk menghitung tingkat kemiripan antar dokumen.

**Insight**: Library `TfidfVectorizer` berperan mengubah teks deskripsi wisata menjadi representasi vektor numerik berbasis frekuensi term, sedangkan `cosine_similarity` mengukur tingkat kemiripan antar vektor tersebut. Kombinasi keduanya menjadi inti dari algoritma Content-Based Filtering yang digunakan dalam proyek ini.

In [ ]:
# 1. Import Library

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, pickle, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## **2. Load Dataset**

Dataset wisata Indonesia dimuat menggunakan `pandas.read_csv()`. Dataset ini berisi informasi lengkap mengenai berbagai tempat wisata di seluruh Indonesia, mencakup nama wisata, provinsi, kota/kabupaten, kategori, deskripsi, dan alamat. Jumlah total data awal ditampilkan untuk memberikan gambaran skala dataset sebelum dilakukan proses selanjutnya.

**Insight**: Mengetahui jumlah data awal sebelum preprocessing memberikan acuan untuk mengevaluasi seberapa banyak data yang digunakan atau terfilter pada langkah-langkah berikutnya. Dataset ini selanjutnya akan difilter khusus untuk wisata yang berlokasi di Provinsi Jawa Barat.

In [ ]:
# 2. Load Dataset

df = pd.read_csv("wisata_indonesia_clean.csv")
print("Jumlah Data Awal :", len(df))

## **3. Data Understanding**

Tahap Data Understanding bertujuan untuk memahami struktur dan kualitas dataset secara menyeluruh. Informasi detail mengenai tipe data tiap kolom (dtypes), jumlah entri non-null, dan keberadaan nilai yang hilang (missing values) diperiksa secara seksama. Ini merupakan langkah krusial sebelum melakukan preprocessing lebih lanjut.

**Insight**: Pemahaman struktur data membantu mengidentifikasi kolom-kolom yang relevan untuk digunakan dalam sistem rekomendasi. Identifikasi missing values sangat penting untuk menentukan strategi penanganan data yang tepat agar kualitas model tidak terganggu oleh data yang tidak lengkap.

In [ ]:
# 3. Data Understanding

print("INFORMASI DATASET")
print("="*50)
print(df.info())

print("MISSING VALUE")
print("="*50)
print(df.isnull().sum())

## **4. Filter Data untuk Jawa Barat**

Data difilter untuk mengambil hanya tempat wisata yang berlokasi di Provinsi Jawa Barat menggunakan metode `str.contains()` yang bersifat case-insensitive. Parameter `na=False` memastikan baris dengan nilai kosong pada kolom provinsi tidak ikut terseleksi secara tidak sengaja. Metode `.copy()` digunakan untuk menghindari `SettingWithCopyWarning` pada operasi-operasi selanjutnya.

**Insight**: Pembatasan cakupan data ke wilayah Jawa Barat membuat sistem rekomendasi lebih terfokus dan relevan secara geografis. Jumlah wisata Jawa Barat yang dihasilkan dari proses filter ini akan menjadi ukuran dataset yang diproses oleh algoritma rekomendasi.

In [ ]:
# 4. Filter Data untuk Jawa Barat

jabar = df[
    df['provinsi'].str.contains(
        'Jawa Barat',
        case=False,
        na=False
    )
].copy()
print("DATA WISATA JAWA BARAT")
print("Jumlah Wisata Jawa Barat :", len(jabar))

## **5. Kategori Wisata**

Kolom kategori baru (`kategori_baru`) dibuat menggunakan fungsi `kategori_wisata()` yang mengklasifikasikan setiap tempat wisata ke dalam tiga kelompok utama berdasarkan keyword dalam kategori aslinya: **Wisata Alam** (gunung, curug, pantai, danau, bukit, hutan, alam, geopark), **Wisata Budaya** (museum, sejarah, budaya), dan **Wisata Rekreasi** untuk semua kategori lainnya.

**Insight**: Standardisasi kategori menjadi tiga kelompok yang lebih umum membantu dalam evaluasi sistem rekomendasi menggunakan metrik Precision@K. Dengan kategori yang lebih terstruktur, relevansi rekomendasi dapat diukur secara kuantitatif berdasarkan kesamaan tipe wisata antara acuan dan hasil rekomendasi.

In [ ]:
# 5. Kategori Wisata

def kategori_wisata(kategori):
    kategori = str(kategori).lower()

    if any(x in kategori for x in [
        'gunung','curug','pantai',
        'danau','bukit','hutan',
        'alam','geopark'
    ]):
        return 'Wisata Alam'

    elif any(x in kategori for x in [
        'museum','sejarah','budaya'
    ]):
        return 'Wisata Budaya'

    return 'Wisata Rekreasi'

jabar['kategori_baru'] = (
    jabar['kategori']
    .apply(kategori_wisata)
)

## **6. Feature Engineering**

Feature Engineering dilakukan dengan menggabungkan kolom `kategori` dan `deskripsi_bersih` menjadi satu kolom baru bernama `content`. Nilai kosong (NaN) pada kedua kolom diisi dengan string kosong menggunakan `fillna('')` sebelum digabungkan, sehingga proses tidak terganggu oleh data yang hilang.

**Insight**: Menggabungkan kategori dengan deskripsi menghasilkan representasi konten yang lebih kaya dan komprehensif. Kategori memberikan konteks tipe wisata secara singkat, sementara deskripsi memberikan detail lengkap tentang karakteristik wisata tersebut, sehingga meningkatkan kualitas kesamaan konten yang akan dihitung oleh algoritma TF-IDF.

In [ ]:
# 6. Feature Engineering

jabar['content'] = (
    jabar['kategori'].fillna('')
    + ' ' +
    jabar['deskripsi_bersih'].fillna('')
)

## **7. Text Cleaning**

Pembersihan teks dilakukan menggunakan fungsi `clean_text()` yang menerapkan tiga tahapan utama: (1) konversi semua huruf menjadi lowercase untuk menghindari duplikasi term yang berbeda hanya karena kapitalisasi, (2) penghapusan semua karakter non-alfabet menggunakan regex `[^a-zA-Z ]`, dan (3) normalisasi spasi berlebih menjadi satu spasi tunggal. Fungsi ini kemudian diterapkan ke seluruh baris pada kolom `content`.

**Insight**: Text cleaning yang konsisten adalah fondasi keberhasilan model berbasis NLP. Dengan menyeragamkan format teks dan menghilangkan noise seperti angka, tanda baca, dan spasi berlebih, representasi vektor TF-IDF yang dihasilkan akan lebih akurat dalam merepresentasikan konten sebenarnya dari setiap tempat wisata.

In [ ]:
# 7. Text Cleaning

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

jabar['content'] = (
    jabar['content']
    .apply(clean_text)
)

## **8. Visualisasi Data**

Distribusi kategori wisata di Jawa Barat divisualisasikan menggunakan bar chart dari library `seaborn`. Visualisasi ini memberikan gambaran komposisi data berdasarkan tiga tipe wisata (Wisata Alam, Wisata Budaya, Wisata Rekreasi) yang ada dalam dataset yang telah difilter dan dikategorikan ulang.

**Insight**: Memahami distribusi kategori wisata sangat penting untuk mendeteksi potensi ketidakseimbangan data (class imbalance). Jika satu kategori sangat mendominasi, rekomendasi yang dihasilkan berpotensi bias ke kategori tersebut. Visualisasi ini membantu dalam memahami representasi setiap tipe wisata dalam dataset secara visual dan intuitif.

In [ ]:
# 8. Visualisasi Data

plt.figure(figsize=(8,5))
sns.countplot(
    data=jabar,
    x='kategori_baru'
)
plt.title('Distribusi Kategori Wisata Jawa Barat')
plt.show()

## **9. Algoritma TF-IDF**

TF-IDF (Term Frequency-Inverse Document Frequency) diterapkan menggunakan `TfidfVectorizer` dari sklearn untuk mengubah teks dalam kolom `content` menjadi matriks numerik. Parameter `max_features=5000` membatasi jumlah fitur untuk efisiensi komputasi dan menghindari overfitting, sedangkan `stop_words='english'` menghilangkan kata-kata umum bahasa Inggris yang tidak memberikan informasi diskriminatif.

**Insight**: Matriks TF-IDF yang dihasilkan merepresentasikan setiap tempat wisata sebagai vektor dalam ruang fitur berdimensi tinggi. Nilai TF-IDF yang tinggi untuk suatu term menunjukkan bahwa term tersebut penting bagi dokumen tertentu namun tidak umum di seluruh dataset, sehingga menjadi pembeda yang baik antar tempat wisata.

In [ ]:
# 9. Algoritma TF-IDF

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(
    jabar['content']
)

print("Shape TF-IDF :", tfidf_matrix.shape)

## **10. Algoritma Cosine Similarity**

Cosine Similarity dihitung dari matriks TF-IDF menggunakan fungsi `cosine_similarity()`. Metode ini mengukur sudut antar vektor dalam ruang fitur, di mana nilai 1 menunjukkan dua tempat wisata identik dan nilai 0 menunjukkan tidak ada kesamaan konten sama sekali. Matriks berukuran n x n dihasilkan, dengan n adalah jumlah data wisata Jawa Barat.

**Insight**: Cosine Similarity dipilih karena tidak sensitif terhadap panjang dokumen, sehingga wisata dengan deskripsi pendek maupun panjang tetap dapat dibandingkan secara adil. Matriks similarity yang dihasilkan menjadi basis utama untuk menemukan wisata-wisata yang paling mirip satu sama lain dalam tahap pembuatan fungsi rekomendasi.

In [ ]:
# 10. Algoritma Cosine Similarity

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print("Shape Cosine Similarity :", cosine_sim.shape)

## **11. Sistem Rekomendasi**

Fungsi `rekomendasi_wisata()` dibangun sebagai inti dari sistem rekomendasi. Fungsi ini menerima nama wisata dan jumlah rekomendasi (`top_n`) sebagai parameter, kemudian mengambil skor similarity dari matriks cosine similarity, mengurutkannya secara menurun, dan mengembalikan `top_n` wisata paling mirip beserta informasi detail kolom nama, kategori, kota/kabupaten, dan alamat.

**Insight**: Pendekatan Content-Based Filtering ini bekerja efisien karena menggunakan lookup langsung dari matriks similarity yang sudah dihitung sebelumnya. Penggunaan `pd.Series` dengan index nama wisata memungkinkan pencarian berbasis nama secara cepat. Nilai similarity wisata itu sendiri dihilangkan dengan slicing `[1:top_n+1]` agar tidak masuk ke dalam daftar rekomendasi.

In [ ]:
# 11. Sistem Rekomendasi

jabar = jabar.reset_index(drop=True)

indices = pd.Series(
    jabar.index,
    index=jabar['nama_wisata']
).drop_duplicates()

def rekomendasi_wisata(nama_wisata, top_n=10):

    idx = indices[nama_wisata]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:top_n+1]

    wisata_idx = [i[0] for i in sim_scores]

    return jabar.iloc[wisata_idx][[
        'nama_wisata',
        'kategori',
        'kategori_baru',
        'kota_kabupaten',
        'alamat'
    ]]

## **12. Pengujian dan Evaluasi**

Sistem diuji menggunakan wisata pertama dalam dataset sebagai acuan. Fungsi `rekomendasi_wisata()` dipanggil untuk menghasilkan 10 rekomendasi teratas. Evaluasi kuantitatif dilakukan menggunakan metrik **Precision@K** yang mengukur proporsi rekomendasi relevan (memiliki `kategori_baru` yang sama dengan wisata acuan) dari total K rekomendasi yang dihasilkan.

**Insight**: Precision@10 memberikan indikasi seberapa akurat sistem dalam merekomendasikan wisata dengan tipe yang serupa. Nilai precision yang mendekati 1.0 menunjukkan bahwa sistem berhasil mengidentifikasi wisata-wisata yang relevan secara konsisten berdasarkan kemiripan konten deskripsi dan kategorinya.

In [ ]:
# 12. Pengujian dan Evaluasi

tempat = jabar['nama_wisata'].iloc[0]

print("\n" + "="*60)
print("WISATA ACUAN")
print("="*60)
print(tempat)

hasil = rekomendasi_wisata(
    tempat,
    top_n=10
)

print("\n" + "="*60)
print("TOP 10 REKOMENDASI")
print("="*60)

display(hasil)

def precision_at_k(
    rekomendasi,
    kategori_target,
    k=10
):
    relevan = rekomendasi[
        rekomendasi['kategori_baru']
        == kategori_target
    ]
    return len(relevan) / k

kategori_target = jabar[
    jabar['nama_wisata'] == tempat
]['kategori_baru'].values[0]

precision = precision_at_k(
    hasil,
    kategori_target,
    10
)

print("\n" + "="*60)
print("HASIL EVALUASI")
print("="*60)
print("Precision@10 =", round(precision,3))

## **13. Visualisasi Hasil dan Simpan Model**

Distribusi kategori dari 10 hasil rekomendasi divisualisasikan menggunakan bar chart untuk mengevaluasi konsistensi tipe wisata antara acuan dan rekomendasinya. Setelah evaluasi selesai, model TF-IDF (`tfidf_model.pkl`) dan matriks Cosine Similarity (`cosine_similarity.pkl`) disimpan ke disk menggunakan `pickle` agar dapat digunakan kembali tanpa proses training ulang.

**Insight**: Penyimpanan model dalam format `.pkl` memungkinkan sistem rekomendasi dapat di-deploy dan digunakan kembali secara efisien. Visualisasi distribusi kategori hasil rekomendasi membantu memverifikasi secara visual bahwa sistem memberikan rekomendasi yang konsisten dan relevan berdasarkan tipe wisata acuan.

In [ ]:
# 13. Visualisasi Hasil dan Simpan Model

plt.figure(figsize=(8,5))
sns.countplot(
    data=hasil,
    x='kategori_baru'
)
plt.title(
    'Distribusi Kategori Hasil Rekomendasi'
)
plt.show()

pickle.dump(
    tfidf,
    open('tfidf_model.pkl','wb')
)

pickle.dump(
    cosine_sim,
    open('cosine_similarity.pkl','wb')
)

print("\n" + "="*60)
print("MODEL BERHASIL DISIMPAN")
print("="*60)

print("tfidf_model.pkl")
print("cosine_similarity.pkl")

## **Kesimpulan**

Proyek ini berhasil membangun sistem rekomendasi wisata Jawa Barat menggunakan pendekatan **Content-Based Filtering** dengan algoritma TF-IDF dan Cosine Similarity. Berikut ringkasan dari setiap tahapan yang telah dilakukan:

1. **Import Library** - Menyiapkan semua dependensi yang dibutuhkan untuk data processing, visualisasi, dan machine learning.
2. **Load Dataset** - Memuat dataset wisata Indonesia dan menampilkan jumlah data awal.
3. **Data Understanding** - Memahami struktur dataset, tipe data, dan kondisi missing values.
4. **Filter Data** - Membatasi dataset pada wisata Provinsi Jawa Barat untuk fokus yang lebih relevan.
5. **Kategori Wisata** - Membuat kategori wisata terstandar (Alam, Budaya, Rekreasi) sebagai label evaluasi.
6. **Feature Engineering** - Menggabungkan kolom kategori dan deskripsi menjadi satu representasi konten.
7. **Text Cleaning** - Membersihkan dan menstandarisasi teks agar siap diproses algoritma NLP.
8. **Visualisasi Data** - Menampilkan distribusi kategori wisata untuk memahami komposisi dataset.
9. **Algoritma TF-IDF** - Mengubah teks menjadi representasi vektor numerik berbasis frekuensi term.
10. **Algoritma Cosine Similarity** - Menghitung kemiripan antar tempat wisata berdasarkan vektor TF-IDF.
11. **Sistem Rekomendasi** - Membangun fungsi rekomendasi yang mengembalikan top-N wisata paling mirip.
12. **Pengujian dan Evaluasi** - Menguji sistem dan mengukur kinerja menggunakan metrik Precision@10.
13. **Simpan Model** - Menyimpan model TF-IDF dan matriks similarity untuk keperluan deployment.

Evaluasi menggunakan **Precision@10** menunjukkan kemampuan sistem dalam merekomendasikan wisata yang relevan berdasarkan kesamaan tipe. Model yang telah dilatih disimpan dalam format `.pkl` untuk memudahkan penggunaan kembali tanpa perlu proses training ulang.